# Test Action Mapping Utilities

This notebook tests the action mapping functions that connect network outputs to MCCFR action keys.

In [7]:
import sys
import os

# Add parent directory to path
current_dir = os.getcwd()
if current_dir.endswith('deep_CFR_vNB_integration'):
    parent_dir = os.path.dirname(current_dir)
else:
    parent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, parent_dir)

import torch
from mccfr import MCCFR
from action_mapping import (
    get_all_network_action_keys,
    map_network_output_to_actions,
    apply_legal_action_mask,
    fill_illegal_action_regrets,
    regrets_dict_to_tensor,
    NETWORK_ACTION_TYPES
)

## Test 1: Get All Network Action Keys

In [8]:
mccfr = MCCFR()
state = mccfr.create_initial_state()
active_player = 0

all_keys = get_all_network_action_keys(state, active_player, mccfr)
print(f"✓ Got {len(all_keys)} action keys in network order:")
for i, key in enumerate(all_keys):
    print(f"  Network[{i}] → {key}")

✓ Got 9 action keys in network order:
  Network[0] → DISCARD_PLACEHOLDER_0
  Network[1] → DISCARD_PLACEHOLDER_1
  Network[2] → DISCARD_PLACEHOLDER_2
  Network[3] → CHECK
  Network[4] → CALL
  Network[5] → FOLD
  Network[6] → RAISE_4
  Network[7] → RAISE_202
  Network[8] → RAISE_400


## Test 2: Map Network Output to Actions

In [9]:
# Create mock network output (9 regrets)
mock_regrets = torch.tensor([-5.0, -3.0, -1.0, 2.0, 5.0, -10.0, 8.0, 12.0, 15.0])
print(f"Mock network output: {mock_regrets.tolist()}")

# Map to action keys
regret_dict = map_network_output_to_actions(mock_regrets, state, active_player, mccfr)
print(f"\n✓ Mapped to action keys:")
for key, regret in regret_dict.items():
    print(f"  {key}: {regret:.2f}")

Mock network output: [-5.0, -3.0, -1.0, 2.0, 5.0, -10.0, 8.0, 12.0, 15.0]

✓ Mapped to action keys:
  CALL: 5.00
  FOLD: -10.00
  RAISE_LARGE: 8.00
  RAISE_MEDIUM: 15.00


## Test 3: Legal Action Masking

In [10]:
# Get legal actions at this state (preflop - no discards legal)
legal_actions = mccfr.get_legal_actions_list(state)
legal_keys = [mccfr.action_to_key(a, state, active_player) for a in legal_actions]
print(f"Legal actions at preflop: {legal_keys}")

# Apply mask
masked_regrets = apply_legal_action_mask(regret_dict, legal_actions, state, active_player, mccfr)
print(f"\n✓ Masked regrets (illegal actions = -1000):")
for key, regret in masked_regrets.items():
    status = "(LEGAL)" if regret > -500 else "(ILLEGAL - masked)"
    print(f"  {key}: {regret:.2f} {status}")

Legal actions at preflop: ['RAISE_MEDIUM', 'RAISE_LARGE', 'RAISE_LARGE', 'FOLD', 'CALL']

✓ Masked regrets (illegal actions = -1000):
  CALL: 5.00 (LEGAL)
  FOLD: -10.00 (LEGAL)
  RAISE_LARGE: 8.00 (LEGAL)
  RAISE_MEDIUM: 15.00 (LEGAL)


## Test 4: Fill Illegal Action Regrets (for training)

In [11]:
# Simulate MCCFR computing regrets for only legal actions
computed_regrets = {
    'FOLD': -10.0,
    'CALL': 5.0,
    'RAISE_SMALL': 15.0
}
print(f"Computed regrets (legal actions only): {computed_regrets}")

# Fill in illegal actions
filled_regrets = fill_illegal_action_regrets(
    computed_regrets, legal_actions, state, active_player, mccfr
)
print(f"\n✓ Filled regrets (all 9 actions):")
for key, regret in filled_regrets.items():
    status = "(computed)" if key in computed_regrets else "(filled - illegal)"
    print(f"  {key}: {regret:.2f} {status}")

Computed regrets (legal actions only): {'FOLD': -10.0, 'CALL': 5.0, 'RAISE_SMALL': 15.0}

✓ Filled regrets (all 9 actions):
  FOLD: -10.00 (computed)
  CALL: 5.00 (computed)
  RAISE_SMALL: 15.00 (computed)


## Test 5: Convert Regrets Dict to Tensor

In [12]:
# Convert filled regrets back to tensor
regrets_tensor = regrets_dict_to_tensor(filled_regrets, state, active_player, mccfr)
print(f"✓ Converted to tensor: {regrets_tensor.tolist()}")
print(f"  Shape: {regrets_tensor.shape}")
print(f"  Should be [9] ✓")

print("\n" + "=" * 70)
print("✓ All Action Mapping Tests Passed!")
print("=" * 70)

✓ Converted to tensor: [-1000.0, -1000.0, -1000.0, -1000.0, 5.0, -10.0, 15.0, -1000.0, -1000.0]
  Shape: torch.Size([9])
  Should be [9] ✓

✓ All Action Mapping Tests Passed!
